In [1]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

Using Python 3.11.11 environment at: /usr
Resolved 168 packages in 302ms                                       
Uninstalled 1 package in 11ms
Installed 1 package in 15ms                                 
 - triton==3.2.0
 + triton==3.3.1
Using Python 3.11.11 environment at: /usr
Resolved 1 package in 4ms                                            
Uninstalled 1 package in 7ms
Installed 1 package in 12ms                                 
 - triton==3.3.1
 + triton==3.2.0
Using Python 3.11.11 environment at: /usr
Audited 1 package in 135ms
Using Python 3.11.11 environment at: /usr
Resolved 3 packages in 1ms                                           
Audited 3 packages in 0.28ms


In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
%%writefile infer14b.py
    import os
    import pandas as pd
    from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
    import torch
    import vllm
    import numpy as np
    from vllm.lora.request import LoRARequest
    import argparse
    from scipy.special import softmax
    
    MODEL_NAME = "/kaggle/input/qwen2.5/transformers/14b-instruct-gptq-int4/1"
    LORA_PATH = "/kaggle/input/lora_14b_gptq_1epoch_r32/keras/default/1"
    
    os.environ["VLLM_USE_V1"] = "0"
    
    # 2. Prepare Model
    
    llm = vllm.LLM(
        MODEL_NAME,
        # quantization='awq',
        quantization='gptq',
        tensor_parallel_size=torch.cuda.device_count(),
        gpu_memory_utilization=0.9,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=4096,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=32
    )
    llm
    
    tokenizer = llm.get_tokenizer()
    tokenizer
    
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
    mclp
    
    # 3. Prepare Prompt
    
    df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv")
    print(df.shape)
    df.head()
    
    SYS_PROMPT = """
    You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
    """
    
    prompts = []
    for i, row in df.iterrows():
        text = f"""
    r/{row.subreddit}
    Rule: {row.rule}
    
    1) {row.positive_example_1}
    Violation: Yes
    
    2) {row.positive_example_2}
    Violation: Yes
    
    3) {row.negative_example_1}
    Violation: No
    
    4) {row.negative_example_2}
    Violation: No
    
    5) {row.body}
    """
        
        messages = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": text}
        ]
    
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        prompts.append(prompt)
    
    df["prompt"] = prompts
    df.shape
    
    print(df.iloc[0]['prompt'])
    outputs = llm.generate(
        prompts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )
    len(outputs)
    logprobs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
    print(logit_matrix.shape)
    logit_matrix.head()
    df = pd.concat([df, logit_matrix], axis=1)
    print(df.shape)
    df.head()
    
    df[['Yes',"No"]].head()
    
    df[['Yes',"No"]] = df[['Yes',"No"]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
    df[['Yes',"No"]].head()
    
    df["pred"] = df["Yes"]
    df['rule_violation'] = df["pred"]
    df[['row_id', 'rule_violation']].to_csv("submission1.csv",index=False)

In [36]:
!head submission1.csv

[rank1]:[W1009 10:09:15.077048630 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank1]:[W1009 10:09:05.068494301 TCPStore.cpp:106] [c10d] sendBytes failed on SocketImpl(fd=48, addr=[fdff:ffff::e0:428b:2ab6:36a7]:45821, remote=[fdff:ffff:0:0:5f00::]:15866): Broken pipe
Exception raised from sendBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:653 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x98 (0x7a0989b785e8 in /usr/local/lib/python3.11/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5ba8bfe (0x7a0972ecdbfe in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5baa458 (0x7a0972ecf458 in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x5babc3e (0x7a0972ed0c3e in /usr/local/lib/python3.11/dist-packages/torch/li

row_id,rule_violation
2029,0.05419872504911194
2030,0.18242551491653522
2031,0.8596637523076225
2032,0.7853085859124002
2033,0.960361159764527
2034,0.04401864563031933
2035,0.9449947124811038
2036,0.06278921429680095
2037,0.03963884023547296


In [37]:
%%writefile check.py
import os
IS_SUB=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
import polars as pl
from scipy.special import softmax

SYS_PRMPT = """
You are an experienced, fair, unbiased moderator. 
Classify whether the comment violates the supplied moderation rule.
- True: the comment breaks the specified rule
- False: the comment is considered safe in relation to the specified rule
Respond only using True or False.
""".strip()

USR_PRMPT_TMPLT = """
[RULE]: {}
[True EXAMPLE]: {}
[False EXAMPLE]: {}
[True EXAMPLE 2]: {}
[False EXAMPLE 2]: {}
[TEST CASE COMMENT]: {}
""".strip()

CHOICES = ['True', 'False']

def chat_formatting(df, tokenizer):
  prompts = []
  for user_content in df['user_content']:
    chat = [
      {'role': 'system', 'content': SYS_PRMPT},
      {'role': 'user', 'content': user_content.strip()},
    ]
    prompt = tokenizer.apply_chat_template(
      chat, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    prompts.append(prompt)
  df = df.with_columns(pl.Series('prompt', prompts))
  return df


if __name__ == '__main__':
  print('importing vllm...')
  import vllm
  from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
  print('loading vllm...')

  llm = vllm.LLM(
    '/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1',
    quantization='awq',
    task='generate',
    tensor_parallel_size=2,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
    enable_prefix_caching=True,
    dtype='half',
    enforce_eager=True,
    disable_log_stats=True,
    disable_custom_all_reduce=True,
  )
  print('building prompts...')
  path = '/kaggle/input/jigsaw-agile-community-rules/test.csv' if IS_SUB else '/kaggle/input/jigsaw-agile-community-rules/train.csv'
  df = (
      pl.read_csv(path)
      .with_columns([pl.col(x).str.replace_all(r'\s+', ' ') for x in ["body", "^.*_example_.*$"]])
      .with_columns(pl.format(USR_PRMPT_TMPLT,  "rule", "positive_example_1", "negative_example_1","positive_example_2", "negative_example_2", "body").alias('user_content'))
  )

  tokenizer = llm.get_tokenizer()
  df = chat_formatting(df, tokenizer)
  prompts = df['prompt'].to_list()
  mclp = MultipleChoiceLogitsProcessor(
    tokenizer,
    choices=CHOICES,
  )
  sampling_params_choice = vllm.SamplingParams(seed=1337, skip_special_tokens=True, max_tokens=1, logits_processors=[mclp], logprobs=len(mclp.choices),)
  outputs = llm.generate(prompts, sampling_params_choice, use_tqdm=True)
  del llm;cleanup()
  logprobs = [
    {lp.decoded_token: lp.logprob for lp in list(lps)}
    for lps in [output.outputs[0].logprobs[0].values() for output in outputs]
  ]
  choices = [max(d, key=d.get) for d in logprobs]
  print('generation end')
  df = df.with_columns(pl.Series('logprobs', logprobs), pl.Series('type', choices))
  print(df.group_by('type').agg(pl.len()).sort(['type']))
  logprobs = df['logprobs'].to_numpy()
  probs = softmax(logprobs, axis=-1)
  sub = df.with_columns(pl.Series('rule_violation', probs[:,0].tolist()))
  sub.select('row_id', 'rule_violation').write_csv('submission2.csv')

Overwriting check.py


In [38]:
!VLLM_USE_V1=0 TORCH_CUDA_ARCH_LIST=7.5 python check.py 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


importing vllm...
loading vllm...
2025-10-09 10:09:21.149435: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760004561.171783    1196 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760004561.178447    1196 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 10-09 10:09:25 [__init__.py:235] Automatically detected platform cuda.


[rank1]:[W1009 10:09:26.093132551 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank1]:[W1009 10:09:16.077376169 TCPStore.cpp:106] [c10d] sendBytes failed on SocketImpl(fd=48, addr=[fdff:ffff::e0:428b:2ab6:36a7]:45821, remote=[fdff:ffff:0:0:5f00::]:15866): Broken pipe
Exception raised from sendBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:653 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x98 (0x7a0989b785e8 in /usr/local/lib/python3.11/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5ba8bfe (0x7a0972ecdbfe in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5baa458 (0x7a0972ecf458 in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x5babc3e (0x7a0972ed0c3e in /usr/local/lib/python3.11/dist-packages/torch/li

`torch_dtype` is deprecated! Use `dtype` instead!
INFO 10-09 10:09:40 [config.py:1604] Using max model len 4096
WARNING 10-09 10:09:41 [config.py:1084] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 10-09 10:09:42 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 10-09 10:09:42 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', speculative_config=None, tokenizer='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=awq, enforce

[rank1]:[W1009 10:09:48.117074987 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank1]:[W1009 10:09:38.107301078 TCPStore.cpp:106] [c10d] sendBytes failed on SocketImpl(fd=48, addr=[fdff:ffff::e0:428b:2ab6:36a7]:45821, remote=[fdff:ffff:0:0:5f00::]:15866): Broken pipe
Exception raised from sendBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:653 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x98 (0x7a0989b785e8 in /usr/local/lib/python3.11/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5ba8bfe (0x7a0972ecdbfe in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5baa458 (0x7a0972ecf458 in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x5babc3e (0x7a0972ed0c3e in /usr/local/lib/python3.11/dist-packages/torch/li

...

[W1009 10:10:05.152074271 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1009 10:10:05.553426593 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3


[rank1]:[W1009 10:10:10.140719144 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank1]:[W1009 10:10:00.126821464 TCPStore.cpp:106] [c10d] sendBytes failed on SocketImpl(fd=48, addr=[fdff:ffff::e0:428b:2ab6:36a7]:45821, remote=[fdff:ffff:0:0:5f00::]:15866): Broken pipe
Exception raised from sendBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:653 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x98 (0x7a0989b785e8 in /usr/local/lib/python3.11/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5ba8bfe (0x7a0972ecdbfe in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5baa458 (0x7a0972ecf458 in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x5babc3e (0x7a0972ed0c3e in /usr/local/lib/python3.11/dist-packages/torch/li

[W1009 10:10:15.162760280 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3


[rank1]:[W1009 10:10:21.155056957 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank1]:[W1009 10:10:11.141102144 TCPStore.cpp:106] [c10d] sendBytes failed on SocketImpl(fd=48, addr=[fdff:ffff::e0:428b:2ab6:36a7]:45821, remote=[fdff:ffff:0:0:5f00::]:15866): Broken pipe
Exception raised from sendBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:653 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x98 (0x7a0989b785e8 in /usr/local/lib/python3.11/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5ba8bfe (0x7a0972ecdbfe in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5baa458 (0x7a0972ecf458 in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x5babc3e (0x7a0972ed0c3e in /usr/local/lib/python3.11/dist-packages/torch/li

[W1009 10:10:25.173284701 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
(VllmWorkerProcess pid=1218) INFO 10-09 10:10:25 [__init__.py:1375] Found nccl from library libnccl.so.2
INFO 10-09 10:10:25 [__init__.py:1375] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=1218) INFO 10-09 10:10:25 [pynccl.py:70] vLLM is using nccl==2.26.2
INFO 10-09 10:10:25 [pynccl.py:70] vLLM is using nccl==2.26.2
INFO 10-09 10:10:25 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[1], buffer_handle=(1, 4194304, 6, 'psm_cb5c3e00'), local_subscribe_addr='ipc:///tmp/81e49b90-697d-4266-affd-2741b5bb08f4', remote_subscribe_addr=None, remote_addr_ipv6=False)
INFO 10-09 10:10:25 [parallel_state.py:1102] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(VllmWorkerProcess pid=1218) INFO 10-09 10:10:25 [parallel_state.py:1102] rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, TP rank 1

In [39]:
sub1= pd.read_csv('submission1.csv')
sub2= pd.read_csv('submission2.csv')
sub1=sub1.merge(sub2.rename(columns={'rule_violation': '32b'}),
                     on='row_id', how='left')
sub1['rule_violation']= sub1['rule_violation'] *.5+ sub1['32b'] *.5
sub1.to_csv('submission.csv',index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'submission2.csv'

In [ ]:
!head submission.csv

[rank1]:[W1009 10:10:32.168490889 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank1]:[W1009 10:10:22.155304081 TCPStore.cpp:106] [c10d] sendBytes failed on SocketImpl(fd=48, addr=[fdff:ffff::e0:428b:2ab6:36a7]:45821, remote=[fdff:ffff:0:0:5f00::]:15866): Broken pipe
Exception raised from sendBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:653 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x98 (0x7a0989b785e8 in /usr/local/lib/python3.11/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5ba8bfe (0x7a0972ecdbfe in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5baa458 (0x7a0972ecf458 in /usr/local/lib/python3.11/dist-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x5babc3e (0x7a0972ed0c3e in /usr/local/lib/python3.11/dist-packages/torch/li